# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [ ]:
%%bash
# Define the environment name
QI_NAME="qiime2-amplicon-2024.10"

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi

### Run the ASV pipeline

In [ ]:

./asv_pipeline.sh --config asv.conf

### Run General Statistics

In [ ]:
mamba activate asv-py
THREADS="$(nproc)"

seqkit stat -a -T -o ../V4_ncbi_output/stats/fastq_stats.tsv -j ${THREADS} ../V4_ncbi_input/*.fastq.gz
seqkit stat -a -T -o ../V4_ncbi_output/stats/fastp_fastqs.tsv -j ${THREADS} ../V4_ncbi_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../V4_ncbi_output/stats/filtered_fastqs.tsv -j ${THREADS} ../V4_ncbi_output/filtered/*.fasta
seqkit stat -a -T -o ../V4_ncbi_output/stats/concat_fastas.tsv -j ${THREADS} ../V4_ncbi_output/concat/concat.fasta

### Run SINA analysis for subsetting V4-V5 to V4 only

In [ ]:
mamba activate sina

sina \
    -i ../V4_ncbi_output/derep/derep.fasta \
    -o ../V4_ncbi_output/derep/derep_SINA.fasta \
    -r ../ref_db/SILVA_138.2_SSURef_NR99_03_07_24_opt.arb \
    -v -p 16 \
    --log-file ../V4_ncbi_output/derep/derep_SINA.log

python parse_sina_log.py \
--log ../V4_ncbi_output/derep/derep_SINA.log \
--output ../V4_ncbi_output/derep/derep_v_regions.tsv \
--verbose

python trim_v_sina.py \
    -m ../V4_ncbi_output/derep/derep_v_regions.tsv \
    -f ../V4_ncbi_output/derep/derep_SINA.fasta \
    -r "V4" -t "V4" \
    -o ../V4_ncbi_output/derep/derep_trim_V4.fasta \
    --threads 12 --batch 1000000




### Run QIIME2 Taxonomic Classifier

In [ ]:
conda activate qiime2-amplicon-2024.10
mkdir -p ../V4_ncbi_output/taxonomy
awk '/^>/ {print; next} {print toupper($0)}' ../V4_ncbi_output/ASVs/ASVs_filtered.fasta > ../V4_ncbi_output/ASVs/ASVs.upper.fasta
python qiime_vs_classifier.py \
  --input-fasta ../V4_ncbi_output/ASVs/ASVs.upper.fasta \
  --ref-taxonomy ../ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs ../ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv ../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output ../V4_ncbi_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Mitomaster, decontamination, mitoDB

In [ ]:
mamba activate asv-py

rm -rf ../V4_ncbi_output/mito
mkdir -p ../V4_ncbi_output/mito/mitomap
rm -rf ../V4_ncbi_output/ASVs/chunks

seqkit split -s 10 -O ../V4_ncbi_output/ASVs/chunks ../V4_ncbi_output/ASVs/ASVs_filtered.fasta

python ./mitomaster.py \
       --data-dir ../V4_ncbi_output/ASVs/chunks/ \
       --output-file ../V4_ncbi_output/mito/mitomap/mitomaster_output.tsv

blastn -query ../V4_ncbi_output/ASVs/ASVs_filtered.fasta \
       -db ../ref_db/mito_ncbi \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ../V4_ncbi_output/mito/mitomap/mito_ncbi.blast6.tsv

blastn -query ../V4_ncbi_output/ASVs/ASVs_filtered.fasta \
       -db ../ref_db/ssu_pipeline_contaminants \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ../V4_ncbi_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv

python ./mito_checker.py \
       --mitomaster-file ../V4_ncbi_output/mito/mitomap/mitomaster_output.tsv \
       --mito-blast ../V4_ncbi_output/mito/mitomap/mito_ncbi.blast6.tsv \
       --silva-tax ../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
       --biof-file ../V4_ncbi_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv \
       --output-dir ../V4_ncbi_output/mito/mitomap/ --overwrite

### Filter ASV count tables

In [ ]:
mkdir -p ../V4_ncbi_output/mito/ASVs
python filter_nontarget.py \
    --count-table ../V4_ncbi_output/ASVs/ASV_filtered.tsv \
    --nontarget-table ../V4_ncbi_output/mito/mitomap/nontarget.master.tsv \
    --taxonomy-table ../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --metadata ../ref_db/asv_cruise_metadata.tsv \
    --output ../V4_ncbi_output/ASVs/ASV_target.tsv \
    --group-col Depth \
    --min-group-size 3 \
    --abundance-threshold 0.005 \
    --save-intermediates \
    --sample-id-col longID


### Build Sankey Diagram

In [ ]:
mkdir -p ../V4_ncbi_output/metadata
mkdir -p ../V4_ncbi_output/mito/metadata
python sankey_builder.py \
    --data-dir ../ \
    --sub-dir V4_ncbi_output \
    --metadata ../ref_db/asv_cruise_metadata.tsv \
    --samp-col sampleid \
    --group1-col Depth \
    --color-col Color \
    --make-labeled \
    --make-unlabeled \
    --verbose \
    --sample-manifest ../ref_db/sample_manifest.tsv


### Plot Metadata

In [ ]:
python plot_metadata.py \
    --data-dir ../ \
    --sub-dir V4_ncbi_output \
    --metadata ../ref_db/asv_cruise_metadata.tsv \
    --group1-col Depth \
    --color-col Color \
    --taxonomy taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --asv-micro ASVs/ASV_target.micro.tsv \
    --asv-mito mito/ASVs/ASV_target.mito.tsv \
    --make-micro --make-mito --verbose \
    --include-rank domain:Bacteria \
    --sample-manifest ../ref_db/sample_manifest.tsv 

python asv_batch_correction.py \
    --data-dir ../V4_ncbi_output \
    --asv ASVs/ASV_final.micro.tsv \
    --asv-meta metadata/ASV_meta_micro.tsv \
    --metadata metadata/metadata_updated_micro.tsv \
    --batch-col plateID \
    --output-dir batch_correction \
    --asv-orientation features_rows \
    --biological-color-col Depth,Month \
    --color-palette-col Color \
    --umap-neighbors 15 \
    --umap-min-dist 0.1 \
    --hdbscan-min-cluster-size 5 \
    --target-clusters 0-100 \
    --verbose

python assign_compartments.py \
    --asv-clr ../V4_ncbi_output/batch_correction/asv_clr_after_correction.tsv \
    --asv-counts ../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --asv-fasta ../V4_ncbi_output/ASVs/ASVs_filtered.fasta \
    --metadata ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --depth-col Depth \
    --month-col Month \
    --biochem-cols "Temperature,Oxygen,Nitrogen Oxides,Nitrate,Nitrite,Ammonium,Nitrous Oxide,Hydrogen Sulfide,Dimethyl Sulfide,Methane,Phosphate,Silicate" \
    --output-dir ../V4_ncbi_output/compartments \
    --use-integrated \
    --verbose

python trajectory_analysis.py \
    --umap-results ../V4_ncbi_output/compartments/compartment_umap_clusters.tsv \
    --asv-data ../V4_ncbi_output/batch_correction/asv_clr_after_correction.tsv \
    --metadata ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --month-col Month \
    --group-cols Depth,Month \
    --color-col Color \
    --top-taxa 10 \
    --output-dir ../V4_ncbi_output/trajectory_analysis \
    --verbose \
    --cluster-color-table ../V4_ncbi_output/compartments/compartment_umap_clusters.tsv

python stratification_anomaly_detection.py \
    --integrated-data ../V4_ncbi_output/compartments/data_integrated.tsv \
    --metadata ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --date-col Date \
    --month-col Month  \
    --year-col Year \
    --depth-col Depth \
    --sample-id-col sampleid \
    --trajectory-summary ../V4_ncbi_output/trajectory_analysis/trajectory_summary.tsv \
    --trajectory-group-col Depth \
    --consensus-threshold 2 \
    --output-dir ../V4_ncbi_output/stratification_timeseries_analysis \
    --biochem-only-data ../ref_db/all_cruise_metadata.tsv 

python aligned_biochem_visuals.py \
  --biochem-only-data ../ref_db/all_cruise_metadata.tsv \
  --output-dir ../V4_ncbi_output/biochem_alignment \
  --month-col Month \
  --year-col Year \
  --depth-col Depth \
  --cruise-col Cruise \
  --group-col Depth \
  --color-col Color \
  --min-coverage 0.51 \
  --n-neighbors 15 \
  --min-dist 0.1





python outlier_checker.py \
  --data-dir ../ \
  --asv V4_ncbi_output/batch_correction/asv_clr_after_correction.tsv \
  --metadata V4_ncbi_output/metadata/metadata_updated_micro.tsv \
  --meta-index-col longID \
  --group-cols Depth \
  --output-dir V4_ncbi_output/outliers_corrected \
  --asv-orientation samples_rows \
  --pre-transformed \
  --transform 'none' \
  --verbose

python collectors_curve.py \
    --counts ../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --meta ../ref_db/asv_cruise_metadata.tsv \
    --sample-id-col sampleid \
    --group-col Depth \
    --color-col Color \
    --out_prefix ../V4_ncbi_output/metadata/collectors_curve \
    --permutations 999 --seed 42

### Plot Upset

In [ ]:
python plot_upset.py \
  --data-dir ../ \
  --subdir V4_ncbi_output \
  --domain micro \
  --sample-id-col sampleid \
  --group-col Depth \
  --color-col Color \
  --skip-venn \
  --formats pdf,svg,png

python bubbleplotter.py \
  --input ../V4_ncbi_output/metadata/ASV_meta_micro.tsv \
  --output-prefix ../V4_ncbi_output/metadata/bubble_plot_asv \
  --no-auto-size \
  --figsize "32,60" \
  --bubble-scale 10

python umap_clustering.py \
  --input ../V4_ncbi_output/metadata/ASV_meta_micro.tsv \
  --output-prefix ../V4_ncbi_output/metadata/umap_clustering \
  --normalize clr \
  --transform sqrt \
  --min-cluster-size 10 \
  --min-samples 5


### Run Alpha and Beta Diversity

In [ ]:
python calc_div.py \
    --micro-table ../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --outdir ../V4_ncbi_output/diversity

python plot_diversity.py \
    --sample-col sampleid \
    --group-col Depth \
    --color-col Color \
    --secondary-col Month \
    --metadata ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --alpha-table ../V4_ncbi_output/diversity/shannon.tsv \
    --distance-bray ../V4_ncbi_output/diversity/bray.tsv \
    --distance-jaccard ../V4_ncbi_output/diversity/jaccard.tsv \
    --output-dir ../V4_ncbi_output/diversity \
    --verbose


### Run indicspecies (R)

In [ ]:
Rscript run_indicspecies.R \
    --asv ../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --meta ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --sample-col sampleid \
    --group-cols Depth,Month \
    --outdir ../V4_ncbi_output

### Plot indicspecies Results

In [ ]:
python plot_indicspecies.py \
    --type-results ../V4_ncbi_output/indicspecies/type_group_indicator_species_results.tsv \
    --status-results ../V4_ncbi_output/indicspecies/status_indicator_species_results.tsv \
    --venn ../V4_ncbi_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --taxonomy ../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --outdir ../V4_ncbi_output/indicspecies/ \
    --type-index  "1=BAL,2=Lung Brush,3=Oral Rinse,4=BAL+Lung Brush,5=BAL+Oral Rinse,6=Lung Brush+Oral Rinse,7=Oral Rinse+BAL+Lung Brush" \
    --status-index "1=Cancer,2=Non-Cancer,3=Cancer+Non-Cancer" \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3" \
    --status-markers "Non-Cancer=D,Cancer=X,Cancer+Non-Cancer=^,not_indicator=o"


python plot_indicspecies.py \
  --group1-results ../V4_ncbi_output/indicspecies/Depth_indicator_species_results.tsv \
  --group2-results ../V4_ncbi_output/indicspecies/Month_indicator_species_results.tsv \
  --group1-label-col Depth \
  --group1-color-col Color \
  --group2-label-col Month \
  --group2-color-col Month_Color \
  --group2-marker-col Month_Marker \
  --outdir ../V4_ncbi_output/indicspecies


### Plot Clustermaps

In [ ]:
python plot_clustermaps.py \
    --asv-meta ../V4_ncbi_output/metadata/ASV_meta_micro.tsv \
    --metadata ../V4_ncbi_output/metadata/metadata_updated_micro.tsv \
    --isa ../V4_ncbi_output/indicspecies/Type_status_ISA_results.tsv \
    --outdir ../V4_ncbi_output/diversity \
    --type-order "Oral Rinse,BAL,Lung Brush" \
    --exclude-types "Skin Brush,Scope Flush" \
    --mito-asv ../V4_ncbi_output/mito/ASVs/ASV_final.mito.tsv \
    --mito-outdir ../V4_ncbi_output/mito/diversity \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3"
    

### Run SPIEC-EASI (R)

In [ ]:
Rscript run_spieceasi.R \
    --counts=../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --outdir=../V4_ncbi_output/spieceasi \
    --force-graphs TRUE

### Graph Network

In [ ]:
python graph_network.py \
    --data-dir ../V4_ncbi_output/ \
    --outdir ../V4_ncbi_output/spieceasi \
    --graph-pos-all ../V4_ncbi_output/spieceasi/spieceasi_network_pos_all.graphml \
    --graph-pos-sub ../V4_ncbi_output/spieceasi/spieceasi_network_pos_thr.graphml \
    --node-features ../V4_ncbi_output/spieceasi/spieceasi_node_features.csv \
    --asv-counts ../V4_ncbi_output/ASVs/ASV_final.micro.tsv \
    --taxonomy ../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --venn ../V4_ncbi_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --type-summary ../V4_ncbi_output/indicspecies/type_group_indicator_species_summary.tsv \
    --status-summary ../V4_ncbi_output/indicspecies/status_indicator_species_summary.tsv
    

Biochem Eigen/Traj analysis

In [ ]:
python env_eigenvectors.py \
  --input ../ref_db/all_cruise_metadata.tsv \
  --outdir ../V4_ncbi_output/env_pca \
  --feature-cols "Oxygen,Nitrate,Nitrite,Nitrous Oxide,Ammonium,Hydrogen Sulfide,Methane,Phosphate,Silicate,Temperature,Salinity,Dimethyl Sulfide,Fe" \
  --pc-selection \
  --impute depth_interp \
  --depth-interp-block-col Cruise \
  --anchor-depths \
  --depth-col Depth \
  --anchored-depth-col Depth_anchored \
  --negatives impute

python env_compartments_o2_soft.py \
  --input ../V4_ncbi_output/env_pca/matrix_cleaned.csv \
  --outdir ../V4_ncbi_output/env_o2_soft_compartments \
  --o2-col Oxygen \
  --T-oxic-dyso 90 \
  --T-dyso-sub 20 \
  --T-sub-anox 1 \
  --softness-s 1 \
  --episodic-smoothing \
  --episodic-block-col Cruise \
  --episodic-sort-cols Depth_anchored \
  --episodic-sticky-prob 0.85 \
  --episodic-apply-to all

python env_compartments_selectk.py \
  --eigenvectors ../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv \
  --pc-keep     ../V4_ncbi_output/env_pca/tables/pc_keep_decision.csv \
  --outdir      ../V4_ncbi_output/env_compartments_selectk \
  --sep $',' \
  --stability-block-col Cruise

python env_compartments_gmm.py \
  --eigenvectors ../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv \
  --pc-keep ../V4_ncbi_output/env_pca/tables/pc_keep_decision.csv \
  --outdir ../V4_ncbi_output/env_compartments_gmm \
  --sep $',' --pc-use-mode keep --standardize-pc-space \
  --episodic-smoothing --random-state 42 --K 6 \
  --matrix-cleaned ../V4_ncbi_output/env_pca/tables/matrix_cleaned_with_sparse.csv

python env_within_gmm_hdbscan.py \
  --eigenvectors   ../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv \
  --assignments  ../V4_ncbi_output/env_compartments_gmm/tables/compartments_assignments_smoothed.csv \
  --outdir       ../V4_ncbi_output/env_compartments_gmm/within_gmm_hdbscan \
  --sep $',' \
  --pc-cols "PC1,PC2,PC3" \
  --standardize-pc-space \
  --hdbscan-min-cluster-size 10 \
  --hdbscan-metric euclidean \
  --min-rows-per-component 10 \
  --high-conf-only \
  --high-conf-maxprob 0.80 \
  --strict-unique-ids

python env_compare_compartments.py \
  --matrix-cleaned ../V4_ncbi_output/env_pca/tables/matrix_cleaned_with_sparse.csv \
  --eigenvectors ../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv \
  --assignments ../V4_ncbi_output/env_compartments_gmm/tables/compartments_assignments_smoothed.csv \
  --outdir ../V4_ncbi_output/env_compare_compartments \
  --sep-matrix $',' \
  --sep-eig $',' \
  --sep-assign $',' \
  --pca-tables-dir ../V4_ncbi_output/env_pca/tables/ \
  --key-mode composite \
  --key-cols "Cruise,Year,Month,Day,Depth" \
  --pc-cols "PC1,PC2,PC3"

python env_compartment_feature_assoc.py \
  --matrix-cleaned ../V4_ncbi_output/env_pca/tables/matrix_cleaned_with_sparse.csv \
  --assignments ../V4_ncbi_output/env_compartments_gmm/tables/compartments_assignments_smoothed.csv \
  --outdir ../V4_ncbi_output/env_compartment_feature_assoc \
  --sep-matrix $',' \
  --sep-assign $',' \
  --bootstrap-B 500 \
  --top-n-each-side 8 \
  --min-n-comp 20 \
  --min-n-rest 50 \
  --depth-adjust

python env_split_o2_by_gmm.py \
    --matrix-cleaned ../V4_ncbi_output/env_pca/tables/matrix_cleaned_with_sparse.csv \
    --eigenvectors ../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv \
    --assignments ../V4_ncbi_output/env_compartments_gmm/tables/compartments_assignments_smoothed.csv \
    --outdir ../V4_ncbi_output/env_o2_split_by_gmm \
    --sep-matrix ',' \
    --sep-eig ',' \
    --sep-assign ',' \
    --key-mode composite \
    --key-cols "Cruise,Year,Month,Day,Depth" \
    --pc-cols "PC1,PC2,PC3" \
    --plots \
    --plot-formats "pdf,png,svg" \
    --umap-embedding ../V4_ncbi_output/env_compare_compartments/tables/umap_embedding.csv \
    --reassign \
    --borderline-mode other_or_low_conf \
    --borderline-max-prob 0.70 \
    --core-min-prob 0.90 \
    --reassign-radius-quantile 0.95 \
    --reassign-min-core-n 30 \
    --min-subcluster-size 20